# Conditional Graph

Two explicit router *nodes*, each emitting labelled conditional edges.  Inputs 4 numbers; routers decide addition vs subtraction based on comparison.

In [1]:
from typing import TypedDict, Literal
from dotenv import load_dotenv

from langgraph.graph import StateGraph, START, END
from IPython.display import display, Markdown

load_dotenv()

True

In [2]:
class State(TypedDict):
    num1: float
    num2: float
    num3: float
    num4: float
    result1: float
    result2: float
    final_result: str

In [3]:
# ---- router node functions (pass-through) ----
def router_1(state: State) -> State:
    return {}


def router_2(state: State) -> State:
    return {}

In [4]:
# ---- conditional-edge routing functions ----
def route_1(state: State) -> Literal["addition_operation", "subtraction_operation"]:
    if state["num1"] > state["num2"]:
        return "addition_operation"
    return "subtraction_operation"


def route_2(state: State) -> Literal["addition_operation2", "subtraction_operation2"]:
    if state["num3"] > state["num4"]:
        return "addition_operation2"
    return "subtraction_operation2"

In [5]:
# ---- compute nodes ----
def add_node(state: State) -> State:
    return {"result1": state["num1"] + state["num2"]}


def subtract_node(state: State) -> State:
    return {"result1": state["num1"] - state["num2"]}


def add_node2(state: State) -> State:
    compute = state["num3"] + state["num4"]
    return {
        "result2": compute,
        "final_result": f"Pair 1 result: {state['result1']}, Pair 2 result: {compute}",
    }


def subtract_node2(state: State) -> State:
    compute = state["num3"] - state["num4"]
    return {
        "result2": compute,
        "final_result": f"Pair 1 result: {state['result1']}, Pair 2 result: {compute}",
    }

In [6]:
graph = StateGraph(State)

# ---- router nodes ----
graph.add_node("router_1", router_1)
graph.add_node("router_2", router_2)

# ---- compute nodes ----
graph.add_node("add_node", add_node)
graph.add_node("subtract_node", subtract_node)
graph.add_node("add_node2", add_node2)
graph.add_node("subtract_node2", subtract_node2)

# ---- layer 1 ----
graph.add_edge(START, "router_1")
graph.add_conditional_edges("router_1", route_1, {
    "addition_operation":   "add_node",
    "subtraction_operation": "subtract_node",
})

# ---- converge to router_2 ----
graph.add_edge("add_node", "router_2")
graph.add_edge("subtract_node", "router_2")

# ---- layer 2 ----
graph.add_conditional_edges("router_2", route_2, {
    "addition_operation2":   "add_node2",
    "subtraction_operation2": "subtract_node2",
})

# ---- terminal ----
graph.add_edge("add_node2", END)
graph.add_edge("subtract_node2", END)

app = graph.compile()

In [7]:
mermaid_code = app.get_graph().draw_mermaid()
display(Markdown(f"```mermaid\n{mermaid_code}\n```"))

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	router_1(router_1)
	router_2(router_2)
	add_node(add_node)
	subtract_node(subtract_node)
	add_node2(add_node2)
	subtract_node2(subtract_node2)
	__end__([<p>__end__</p>]):::last
	__start__ --> router_1;
	add_node --> router_2;
	router_1 -. &nbsp;addition_operation&nbsp; .-> add_node;
	router_1 -. &nbsp;subtraction_operation&nbsp; .-> subtract_node;
	router_2 -. &nbsp;addition_operation2&nbsp; .-> add_node2;
	router_2 -. &nbsp;subtraction_operation2&nbsp; .-> subtract_node2;
	subtract_node --> router_2;
	add_node2 --> __end__;
	subtract_node2 --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [8]:
tests = {
    "add + add":       (10, 3, 8, 1),
    "add + subtract":  (10, 3, 1, 8),
    "subtract + add":  (3, 10, 8, 1),
    "subtract + subt": (3, 10, 1, 8),
}

for label, (n1, n2, n3, n4) in tests.items():
    r = app.invoke({"num1": n1, "num2": n2, "num3": n3, "num4": n4})
    print(f"{label}:\n  {r['final_result']}\n")

add + add:
  Pair 1 result: 13, Pair 2 result: 9

add + subtract:
  Pair 1 result: 13, Pair 2 result: -7

subtract + add:
  Pair 1 result: -7, Pair 2 result: 9

subtract + subt:
  Pair 1 result: -7, Pair 2 result: -7

